# Experimento 2: Prueba con un solo modelo

**TFG: VisionLeaderUFV** | Prueba con **google/paligemma2-3b-mix-224**.

- **Imagenes**: sube manualmente la carpeta de imagenes a Colab en `/content/building_sign`.
- **Resultados**: se guardan en Colab en `/content/results_ex2/` (todo dentro de esa carpeta).
- Opcional: montar Drive si tienes el config en `tfg/config/`; si no, sube `ufv_building_sign_expected_answers.json` a `/content/config/`.

## 1. Instalacion

In [ ]:
!pip install -q transformers>=4.40.0 accelerate torch torchvision pillow huggingface_hub


## 2. Montar Google Drive (Colab, opcional)

Solo si tienes el archivo de config en Drive (`tfg/config/ufv_building_sign_expected_answers.json`). Si subes el JSON a `/content/config/`, puedes saltar esta celda.

In [ ]:
from google.colab import drive
import os
from pathlib import Path
drive.mount("/content/drive")
# Ruta a tfg (solo para cargar config si lo tienes en Drive)
for base in ["/content/drive/MyDrive/tfg", "/content/drive/Mi unidad/tfg"]:
    if os.path.exists(base):
        BASE_DIR = Path(base)
        break
else:
    BASE_DIR = None
print("BASE_DIR (para config):", BASE_DIR)


## 3. Login Hugging Face

In [ ]:
from huggingface_hub import login
login()


## 4. Modelo (un solo modelo)

In [ ]:
MODEL_ID = "google/paligemma2-3b-mix-224"
print(f"Modelo: {MODEL_ID}")


## 5. Cargar modelo

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from huggingface_hub import get_token
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
hf_token = get_token()
processor = AutoProcessor.from_pretrained(MODEL_ID, token=hf_token)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map="auto", token=hf_token).eval()
print("Modelo cargado.")


## 6. Funcion de inferencia

In [ ]:
from PIL import Image
import time
def ask_paligemma(image, question, max_tokens=100):
    if not question.strip().startswith("<image>"):
        question = "<image>\n" + question
    inputs = processor(text=question, images=image, return_tensors="pt").to(device, dtype=dtype)
    input_len = inputs["input_ids"].shape[-1]
    start = time.time()
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    t = time.time() - start
    answer = processor.decode(output[0][input_len:], skip_special_tokens=True)
    return {"answer": answer.strip(), "inference_time": t}
print("OK")


## 7. Cargar dataset

In [ ]:
import json
from pathlib import Path
import os

# Imagenes: carpeta que subes manualmente a Colab en /content/building_sign
DATA_PATH = Path("/content/building_sign")
# Config: primero en /content/config/, luego en Drive si montaste
_candidates = [Path("/content/config/ufv_building_sign_expected_answers.json")]
if "BASE_DIR" in dir() and BASE_DIR is not None:
    _candidates.append(BASE_DIR / "config" / "ufv_building_sign_expected_answers.json")
EXPECTED_ANSWERS_PATH = next((p for p in _candidates if p.exists()), None)

print("DATA_PATH:", DATA_PATH)
print("DATA_PATH existe:", DATA_PATH.exists())
if DATA_PATH.exists():
    ejemplos = list(DATA_PATH.glob("*.JPEG"))[:5] or list(DATA_PATH.glob("*.jpeg"))[:5]
    print("Ejemplo imagenes:", [p.name for p in ejemplos])
print("Config existe:", EXPECTED_ANSWERS_PATH is not None and EXPECTED_ANSWERS_PATH.exists())

if EXPECTED_ANSWERS_PATH and EXPECTED_ANSWERS_PATH.exists():
    with open(EXPECTED_ANSWERS_PATH, "r", encoding="utf-8") as f:
        _raw = json.load(f)
    EXPECTED_ANSWERS = {k: v for k, v in _raw.items() if not k.startswith("_")}
    EXPECTED_ANSWERS_LOWER = {k.lower(): v for k, v in EXPECTED_ANSWERS.items()}
    PROMPTS_CONFIG = _raw.get("_prompts", {})
    LANDMARKS_REFERENCE = _raw.get("_landmarks_reference", {})
    LANDMARKS_REFERENCE_LOWER = {k.lower(): v for k, v in LANDMARKS_REFERENCE.items()}
    print(f"Expected answers: {len(EXPECTED_ANSWERS)} imagenes")
    print(f"Prompts: {list(PROMPTS_CONFIG.keys())}")
    print(f"Landmarks ref: {len(LANDMARKS_REFERENCE)} imagenes")
else:
    EXPECTED_ANSWERS = {}
    EXPECTED_ANSWERS_LOWER = {}
    PROMPTS_CONFIG = {}
    LANDMARKS_REFERENCE = {}
    LANDMARKS_REFERENCE_LOWER = {}
    print("Config no encontrado. Sube ufv_building_sign_expected_answers.json a /content/config/ o monta Drive con tfg/config/.")

image_files = []
if DATA_PATH.exists():
    seen = set()
    for ext in ["*.JPEG", "*.jpeg", "*.jpg", "*.JPG"]:
        for img_file in sorted(DATA_PATH.glob(ext), key=lambda p: p.name):
            key = img_file.name.lower()
            if key not in seen and key in EXPECTED_ANSWERS_LOWER:
                seen.add(key)
                image_files.append(img_file)
    image_files.sort(key=lambda p: p.name)
    print(f"Imagenes en {DATA_PATH}: {len(image_files)}")
else:
    print("Carpeta no encontrada:", DATA_PATH, "- Sube la carpeta building_sign a /content/building_sign")
print(f"Total a evaluar: {len(image_files)}")


## 8. Funciones de extraccion y keyword landmarks

In [ ]:
import re

# ============================================================
# FUNCIONES DE EXTRACCION, PROMPTS Y KEYWORD LANDMARKS
# ============================================================

def get_prompt_for_image(image_name: str) -> str:
    """Devuelve el prompt segun el numero de imagen (01-06 building+floor, 07 building, 08-10 floor)."""
    m = re.search(r"(\d{2})", image_name)
    num = int(m.group(1)) if m else 0
    if 1 <= num <= 6:
        key = "building_and_floor"
    elif num == 7:
        key = "building_only"
    elif 8 <= num <= 10:
        key = "floor_only"
    else:
        key = "building_and_floor"
    return PROMPTS_CONFIG.get(key, "Extract building and floor from this sign.")


def extract_building_code(text: str, expected_acceptable: list = None) -> str:
    """Extrae codigo de edificio. Si expected_acceptable, usa para coincidencia flexible."""
    text_upper = text.upper().strip()
    text_lower = text.strip().lower()
    if expected_acceptable:
        for alt in expected_acceptable:
            if alt and alt.upper() in text_upper:
                return alt.upper() if len(alt) <= 2 else "H"
            if alt and alt.lower() in text_lower:
                return "H"
    known_codes = ["EPS", "CD", "H", "B"]
    for code in known_codes:
        if code in text_upper:
            return code
    simple_pattern = r"^(EPS|CD|H|B)\s+-?\d+"
    match = re.match(simple_pattern, text_upper)
    if match:
        return match.group(1)
    structured_pattern = r"building\s*[:-]?\s*(EPS|CD|H|B)"
    match = re.search(structured_pattern, text_upper)
    if match:
        return match.group(1)
    return None


# Plantas validas en carteles UFV (evita confundir "11-12" o "1.1" con la planta)
_VALID_FLOORS = {"-2", "-1", "0", "1", "2"}


def extract_floor_number(text: str) -> str:
    """Extrae numero de planta. Prioriza numeros validos (-2,-1,0,1,2) para no confundir con rangos tipo PLANTA 11-12."""
    text_clean = text.strip()
    # 1) floor/planta/level + numero, solo si es planta valida
    for pattern in [
        r"floor\s*[:-]?\s*(-?\d+)",
        r"planta\s*[:-]?\s*(-?\d+)",
        r"level\s*[:-]?\s*(-?\d+)",
    ]:
        match = re.search(pattern, text_clean.lower())
        if match and match.group(1) in _VALID_FLOORS:
            return match.group(1)
    # 2) Linea que sea solo planta valida (ej. "-1" en "UFV\n-1\nPLANTA 11-12")
    for line in text_clean.replace("\r", "\n").split("\n"):
        line = line.strip()
        if line in _VALID_FLOORS:
            return line
        m = re.match(r"^(-?\d+)\s*[:\-]?\s*", line)
        if m and m.group(1) in _VALID_FLOORS:
            return m.group(1)
    # 3) Building + numero, solo si valido
    simple_pattern = r"(?:EPS|CD|H|B|BUILDING|EDIFICIO)\s+(-?\d+)\b"
    match = re.search(simple_pattern, text_clean.upper())
    if match and match.group(1) in _VALID_FLOORS:
        return match.group(1)
    end_number_pattern = r"\s+(-?\d+)\s*$"
    match = re.search(end_number_pattern, text_clean)
    if match and match.group(1) in _VALID_FLOORS:
        return match.group(1)
    if text_clean in _VALID_FLOORS:
        return text_clean
    return None


def is_building_match(predicted: str, expected: str, acceptable: list = None) -> bool:
    """True si predicted coincide con expected (o con alguna variante en acceptable)."""
    if not predicted:
        return False
    pred_upper = predicted.upper()
    exp_upper = expected.upper() if expected else ""
    if pred_upper == exp_upper:
        return True
    if acceptable:
        for alt in acceptable:
            if alt and pred_upper == alt.upper():
                return True
    return False


def calculate_landmarks_keyword_metrics(results: list, landmarks_ref: dict) -> dict:
    """Precision, Recall, F1 por keywords. Normaliza acentos y acepta multi-palabra (todas las palabras del keyword en la prediccion)."""
    import unicodedata
    def _norm(s):
        s = (s or "").lower().strip()
        n = unicodedata.normalize("NFD", s)
        return "".join(c for c in n if unicodedata.category(c) != "Mn")
    def _matches(kw, pred_norm):
        kw_n = _norm(kw)
        if not kw_n: return False
        words = kw_n.split()
        if len(words) <= 1: return kw_n in pred_norm
        return all(w in pred_norm for w in words)
    total_ref_found = 0
    total_ref = 0
    total_lines = 0
    total_lines_with_ref = 0
    for r in results:
        img_key = r["image"].lower() if isinstance(r["image"], str) else r["image"]
        if img_key not in landmarks_ref or not landmarks_ref[img_key]:
            continue
        text = (r.get("prediction") or "")
        pred_norm = _norm(text)
        keywords = [k.strip() for k in landmarks_ref[img_key] if k and isinstance(k, str)]
        ref_found = sum(1 for k in keywords if k and _matches(k, pred_norm))
        total_ref_found += ref_found
        total_ref += len(keywords)
        lines = [s.strip() for s in text.splitlines() if s.strip()]
        total_lines += len(lines)
        total_lines_with_ref += sum(1 for line in lines if any(_matches(k, _norm(line)) for k in keywords))
    if total_ref == 0:
        return {"precision": None, "recall": None, "f1": None}
    recall = total_ref_found / total_ref
    precision = total_lines_with_ref / total_lines if total_lines > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}


print("Funciones de extraccion y keyword landmarks listas!")


## 9. Ejecutar evaluacion (un solo modelo)

In [ ]:
# Evaluacion: un solo modelo. Resultados en Colab: /content/results_ex2
from datetime import datetime
from pathlib import Path

RESULTS_DIR = Path("/content/results_ex2")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_name = MODEL_ID.split("/")[-1]

print("="*70)
print(f"Evaluando: {MODEL_ID}")
print(f"Imagenes: {len(image_files)}")
print("="*70)

if len(image_files) == 0:
    print("AVISO: No hay imagenes. Sube la carpeta building_sign a /content/building_sign y el config a /content/config/ (o monta Drive con tfg/config/).")

results = []
for img_file in image_files:
    img_name = img_file.name
    image = Image.open(img_file).convert("RGB")
    prompt_bf = get_prompt_for_image(img_name)
    result_bf = ask_paligemma(image, prompt_bf)
    rec = {
        "image": img_name,
        "prediction": result_bf["answer"],
        "expected": EXPECTED_ANSWERS_LOWER.get(img_name.lower(), {}),
        "inference_time": result_bf["inference_time"]
    }
results.append(rec)
    print(f"  [{len(results)}/{len(image_files)}] {img_name} -> b/f: {result_bf['answer'][:40]}...")

# Metricas building/floor/location
building_correct = building_total = 0
floor_correct = floor_total = 0
location_correct = location_total = 0
for r in results:
    img_name, prediction, expected = r["image"], r["prediction"], r["expected"]
    exp_building = expected.get("building")
    exp_floor = expected.get("floor")
    building_acceptable = expected.get("building_acceptable", [])
    pred_building = extract_building_code(prediction, building_acceptable)
    pred_floor = extract_floor_number(prediction)
    building_ok = None
    if exp_building is not None:
        building_total += 1
        building_ok = is_building_match(pred_building, exp_building, building_acceptable)
        if building_ok:
            building_correct += 1
    if exp_floor is not None:
        floor_total += 1
        floor_ok = bool(pred_floor and pred_floor == exp_floor)
        if floor_ok:
            floor_correct += 1
    else:
        floor_ok = False
    if exp_building is not None and exp_floor is not None:
        location_total += 1
        if building_ok and floor_ok:
            location_correct += 1

building_accuracy = building_correct / building_total if building_total > 0 else None
floor_accuracy = floor_correct / floor_total if floor_total > 0 else None
location_accuracy = location_correct / location_total if location_total > 0 else None

if len(results) == 0:
    avg_time = 0.0
    total_time = 0.0
    landmarks_kw = {"recall": None, "f1": None, "precision": None}
    print("No hay imagenes evaluadas. Sube building_sign a /content/building_sign y el config a /content/config/.")
else:
    avg_time = sum(x["inference_time"] for x in results) / len(results)
    total_time = sum(x["inference_time"] for x in results)
    landmarks_kw = calculate_landmarks_keyword_metrics(results, LANDMARKS_REFERENCE_LOWER)

# Guardar JSON
model_dir = RESULTS_DIR / model_name
model_dir.mkdir(parents=True, exist_ok=True)
with open(model_dir / f"{model_name}_responses_{timestamp}.json", "w", encoding="utf-8") as f:
    json.dump({"model_id": MODEL_ID, "timestamp": timestamp, "responses": [{"image": x["image"], "prediction": x["prediction"], "inference_time": x["inference_time"]} for x in results]}, f, indent=2, ensure_ascii=False)
with open(model_dir / f"{model_name}_metrics_{timestamp}.json", "w", encoding="utf-8") as f:
    json.dump({
        "model_id": MODEL_ID, "timestamp": timestamp,
        "building_accuracy": building_accuracy, "floor_accuracy": floor_accuracy, "location_accuracy": location_accuracy,
        "building_correct": building_correct, "building_total": building_total,
        "floor_correct": floor_correct, "floor_total": floor_total,
        "location_correct": location_correct, "location_total": location_total,
        "avg_inference_time_s": avg_time, "total_inference_time_s": total_time,
        "landmarks_keyword_recall": landmarks_kw["recall"], "landmarks_keyword_f1": landmarks_kw["f1"], "landmarks_keyword_precision": landmarks_kw["precision"]
    }, f, indent=2)
print(f"Guardado: {model_dir}")

# Imprimir metricas
print("\n" + "="*70)
print("METRICAS")
print("="*70)
if len(results) == 0:
    print("(Sin imagenes evaluadas)")
else:
    if building_accuracy is not None:
        print(f"Building Accuracy: {building_accuracy:.2%} ({building_correct}/{building_total})")
    print(f"Floor Accuracy: {floor_accuracy:.2%} ({floor_correct}/{floor_total})")
    if location_accuracy is not None:
        print(f"Location Accuracy: {location_accuracy:.2%} ({location_correct}/{location_total})")
    print(f"Landmarks Keyword Recall: {landmarks_kw['recall']}")
    print(f"Landmarks Keyword F1: {landmarks_kw['f1']}")
    print(f"Landmarks Keyword Precision: {landmarks_kw['precision']}")
    print(f"Tiempo medio: {avg_time:.2f}s | Total: {total_time:.2f}s")
print("="*70)
